In [1]:
import paddle
print(paddle.utils.run_check())

/media/data3/users/huytq/miniconda3/envs/huy/lib/python3.11/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


Running verify PaddlePaddle program ... 


/media/data3/users/huytq/miniconda3/envs/huy/lib/python3.11/site-packages/paddle/pir/math_op_patch.py:241: UserWarning: Tensor do not have 'place' interface for pir graph mode, try not to use it. None will be returned.
  warnings.warn(
I0316 02:35:46.326020 1807799 pir_interpreter.cc:1529] New Executor is Running ...
I0316 02:35:46.326313 1807799 pir_interpreter.cc:1552] pir interpreter is running by multi-thread mode ...


PaddlePaddle works well on 1 CPU.
PaddlePaddle is installed successfully! Let's start deep learning with PaddlePaddle now.
None


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
import time

import pypdfium2 as pdfium

BASE_DIR = Path.cwd()
PDF_PATH = BASE_DIR / "data_3" / "06_ Quy định ngoại ngữ từ K70_chính quy_final.pdf"
SCRIPT_PATH = BASE_DIR / "ppstructurev3_to_md.py"
PAGE_IMG_DIR = BASE_DIR / "debug"
PAGE_OUT_DIR = BASE_DIR / "outputs_ppstructurev3_md" / "Test_pages"

PAGE_NUMBERS = None
SCALE = 2.0

if not PDF_PATH.exists():
    raise FileNotFoundError(f"Khong tim thay file PDF: {PDF_PATH}")
if not SCRIPT_PATH.exists():
    raise FileNotFoundError(f"Khong tim thay script: {SCRIPT_PATH}")

if PAGE_IMG_DIR.exists():
    shutil.rmtree(PAGE_IMG_DIR)
PAGE_IMG_DIR.mkdir(parents=True, exist_ok=True)
PAGE_OUT_DIR.mkdir(parents=True, exist_ok=True)

pdf = pdfium.PdfDocument(str(PDF_PATH))
total_pages = len(pdf)
selected_pages = PAGE_NUMBERS or list(range(1, total_pages + 1))

print(f"Tong so trang: {total_pages}")
print(f"Se chay cac trang: {selected_pages}")

page_timings = []
failed_pages = []

for page_number in selected_pages:
    page_index = page_number - 1
    page = pdf[page_index]
    bitmap = page.render(scale=SCALE)
    pil_image = bitmap.to_pil()

    page_img_path = PAGE_IMG_DIR / f"Test_page_{page_number:03d}.png"
    pil_image.save(page_img_path)

    start_time = time.time()
    cmd = [
        sys.executable,
        str(SCRIPT_PATH),
        "--input", str(page_img_path),
        "--output_dir", str(PAGE_OUT_DIR),
        "--lang", "vi",
        "--device", "cpu",
        "--vietocr_device", "cpu",
        "--disable_mkldnn",
        "--page_header",
    ]

    print(f"\nDang xu ly trang {page_number}: {page_img_path.name}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - start_time

    if result.returncode == 0:
        print(f"  OK - {elapsed:.1f}s")
        page_timings.append((page_number, elapsed))
    else:
        print(f"  FAIL - {elapsed:.1f}s")
        failed_pages.append(page_number)
        if result.stderr:
            print(result.stderr[-1500:])
        elif result.stdout:
            print(result.stdout[-1500:])

print("\nTong ket theo trang")
for page_number, elapsed in page_timings:
    print(f"- Trang {page_number}: {elapsed:.1f}s")

if failed_pages:
    print("\nTrang loi:")
    for page_number in failed_pages:
        print(f"- Trang {page_number}")

print(f"\nAnh tung trang nam o: {PAGE_IMG_DIR}")
print(f"Markdown tung trang nam o: {PAGE_OUT_DIR}")

In [ ]:
from pathlib import Path
import subprocess
import sys
import time

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "input"
MARKER_OUT_ROOT = BASE_DIR / "outputs"
MARKER_OUT_ROOT.mkdir(parents=True, exist_ok=True)

PAGE_RANGE = None
TIMEOUT_SECONDS = 1800 
MAX_FILES = None 

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Khong tim thay thu muc: {DATA_DIR}")

pdf_files = sorted(DATA_DIR.rglob("*.pdf"))
if MAX_FILES is not None:
    pdf_files = pdf_files[:MAX_FILES]

if not pdf_files:
    raise FileNotFoundError(f"Khong tim thay file PDF trong: {DATA_DIR}")

marker_single = Path(sys.executable).parent / "marker_single"
if not marker_single.exists():
    print("Khong tim thay marker_single, dang cai marker-pdf...")
    install_cmd = [sys.executable, "-m", "pip", "install", "marker-pdf"]
    install_result = subprocess.run(install_cmd, capture_output=True, text=True)
    if install_result.returncode != 0:
        print(install_result.stdout[-2000:])
        print(install_result.stderr[-2000:])
        raise RuntimeError("Cai marker-pdf that bai")
    marker_single = Path(sys.executable).parent / "marker_single"
    if not marker_single.exists():
        raise FileNotFoundError(f"Da cai marker-pdf nhung van khong tim thay marker_single: {marker_single}")

print(f"Tim thay {len(pdf_files)} file PDF trong {DATA_DIR}")

success_files = []
failed_files = []

for i, pdf_path in enumerate(pdf_files, 1):
    cmd = [
        str(marker_single),
        str(pdf_path),
        "--output_dir", str(MARKER_OUT_ROOT),
        "--output_format", "markdown",
        "--disable_multiprocessing",
    ]
    if PAGE_RANGE:
        cmd.extend(["--page_range", PAGE_RANGE])

    print(f"\n[{i}/{len(pdf_files)}] Dang xu ly: {pdf_path.name}")
    print("Lenh:", " ".join(cmd))

    start = time.time()
    try:
        completed = subprocess.run(cmd, text=True, timeout=TIMEOUT_SECONDS)
        return_code = completed.returncode
    except subprocess.TimeoutExpired:
        elapsed = time.time() - start
        print(f"  TIMEOUT sau {elapsed:.1f}s")
        failed_files.append((pdf_path, "timeout"))
        continue

    elapsed = time.time() - start
    print(f"  Thoi gian: {elapsed:.1f}s | Return code: {return_code}")

    base_name = pdf_path.stem
    md_path = MARKER_OUT_ROOT / base_name / f"{base_name}.md"

    if return_code == 0 and md_path.exists():
        print(f"  OK -> {md_path}")
        success_files.append((pdf_path, md_path))
    else:
        print(f"  FAIL -> khong thay markdown tai {md_path}")
        failed_files.append((pdf_path, "missing markdown"))

print("\nTong ket")
print(f"- Thanh cong: {len(success_files)}")
print(f"- That bai: {len(failed_files)}")

if failed_files:
    print("\nDanh sach loi:")
    for pdf_path, reason in failed_files:
        print(f"- {pdf_path} | {reason}")

\- TEDS (Tree Edit Distance for Structure): Dành riêng cho bảng biểu. Nó không chỉ xem chữ trong ô đúng không mà còn xem cấu trúc hàng/cột có bị lệch không <br>
\- TED (Text Edit Distance) sau khi Normalize: Họ dùng các regex để biến tất cả bullet point về cùng một loại, xóa khoảng trắng thừa, rồi mới đo CER/WER <br>
\- BLEU/ROUGE-L: Đo độ tương đồng về ngữ nghĩa. Nếu sai một từ nối nhưng công thức toán đúng, điểm vẫn sẽ cao <br>
\- LaTeX Similarity: Riêng với công thức toán, parse sang cây cú pháp để so sánh

In [ ]:
from pathlib import Path
from collections import Counter
from io import StringIO
import math
import re

import pandas as pd

try:
    import jiwer
except Exception:
    jiwer = None

try:
    from nltk.translate.meteor_score import meteor_score
except Exception:
    meteor_score = None


class OCRBenchEvaluator:
    def __init__(self, lowercase=True, weights=None):
        self.lowercase = lowercase
        self.weights = weights or {"long_text": 0.5, "table": 0.25, "formula": 0.25}

    def evaluate_files(self, pred_path, gt_path):
        pred_text = Path(pred_path).read_text(encoding="utf-8", errors="ignore")
        gt_text = Path(gt_path).read_text(encoding="utf-8", errors="ignore")
        return self.evaluate_texts(pred_text, gt_text)

    def evaluate_texts(self, pred_text, gt_text):
        # normalize toàn document 
        pred = self._normalize_document(pred_text)
        gt = self._normalize_document(gt_text)

        # Lấy công thức rồi chấm formula_cer, sau đó loại bỏ công thức khỏi text để chấm các metric khác
        pred_formulas = self._extract_formulas(pred)
        gt_formulas = self._extract_formulas(gt)
        formula_cer = self._formula_cer(pred_formulas, gt_formulas)

        # Lấy bảng rồi chấm TEDS_Table
        pred_tables = self._extract_tables(pred)
        gt_tables = self._extract_tables(gt)
        pred_cells = self._flatten_cells(pred_tables)
        gt_cells = self._flatten_cells(gt_tables)
        table_score = self._table_score(pred_cells, gt_cells)

        # Loại bỏ công thức và bảng để chấm các metric text-only
        pred_text_only = self._extract_text_only(pred)
        gt_text_only = self._extract_text_only(gt)

        # Tính các metric text-only
        cer = self._cer(pred_text_only, gt_text_only)
        wer = self._wer(pred_text_only, gt_text_only)
        bleu = self._bleu(pred_text_only, gt_text_only)
        meteor = self._meteor(pred_text_only, gt_text_only)
        f1 = self._token_f1(pred_text_only, gt_text_only)
        ned = self._ned(pred_text_only, gt_text_only)
        long_text_hybrid = (bleu + meteor + f1 + ned) / 4.0

        final_score = self._final_score(long_text_hybrid, table_score, formula_cer)

        result = {
            "cer": cer, 
            "wer": wer,
            "teds_table": table_score,
            "formula_cer": formula_cer,
            "bleu": bleu, 
            "meteor": meteor,
            "f1": f1,
            "ned": ned,
            "long_text_hybrid": long_text_hybrid,
            "final_score": final_score,
            "pred_table_count": len(pred_tables),
            "gt_table_count": len(gt_tables),
            "pred_table_cells": len(pred_cells),
            "gt_table_cells": len(gt_cells),
        }

        summary_df = pd.DataFrame(
            [
                {
                    "CER": cer, # Mức sai khác ký tự 
                    "WER": wer, # Mức sai khác từ
                    "TEDS_Table": table_score if table_score is not None else pd.NA,
                    "Formula-CER": formula_cer,
                    "BLEU": bleu, # Mức khớp cụm từ + thứ tự 
                    "METEOR": meteor, # Mức khớp nội dung 
                    "F1": f1, # Mức khớp token (chỉ cần token nào đó của GT xuất hiện trong pred là được tính đúng)
                    "NED": ned, # Mức sai khác nội dung (đo bằng edit distance trên token)
                    "LongTextHybrid": long_text_hybrid, # Điểm tổng hợp cho text dài 
                    "Final_Score": final_score,
                    "Pred-Tables": len(pred_tables),
                    "GT-Tables": len(gt_tables),
                    "Pred-Cells": len(pred_cells),
                    "GT-Cells": len(gt_cells),
                }
            ]
        )
        return result, summary_df

    # Normalize
    def _normalize_document(self, s):
        s = s.replace("\r", "\n") # Chuẩn hóa xuống dòng
        if self.lowercase:
            s = s.lower()
        s = re.sub(r"<\s*br\s*/?\s*>", " ", s, flags=re.IGNORECASE) # Thay thẻ <br> bằng khoảng trắng
        s = re.sub(r"(?m)^\s*[*+]\s+", "- ", s) # Chuẩn hóa bullet marker * hoặc + thành -
        s = re.sub(r"(?m)^\s*-\s+", "- ", s) # Chuẩn hóa bullet marker - (nếu có khoảng trắng thừa) thành đúng format -
        return s

    # Formula 
    def _extract_formulas(self, s): 
        block = re.findall(r"\$\$(.+?)\$\$", s, flags=re.DOTALL) 
        s_no_block = re.sub(r"\$\$(.+?)\$\$", " ", s, flags=re.DOTALL) 
        inline = re.findall(r"\$(.+?)\$", s_no_block, flags=re.DOTALL) 
        return [re.sub(r"\s+", "", x).strip().lower() for x in (block + inline) if x.strip()] # 

    # Table 
    def _norm_cell(self, x):
        x = re.sub(r"<[^>]+>", " ", x)
        x = re.sub(r"\s+", " ", x)
        return x.strip()

    def _extract_markdown_tables(self, s):
        lines = s.splitlines()
        tables, i = [], 0

        def is_sep(line):
            return bool(re.fullmatch(r"\s*\|?\s*[:\-]+(?:\s*\|\s*[:\-]+)+\s*\|?\s*", line))

        while i < len(lines):
            if "|" not in lines[i]:
                i += 1
                continue
            j, chunk = i, []
            while j < len(lines) and "|" in lines[j]:
                chunk.append(lines[j])
                j += 1
            if any(is_sep(c) for c in chunk):
                rows = []
                for r in chunk:
                    if is_sep(r):
                        continue
                    cells = [self._norm_cell(p) for p in r.strip().strip("|").split("|")]
                    if len(cells) >= 2:
                        rows.append(cells)
                if rows:
                    tables.append(rows)
            i = j
        return tables

    def _extract_html_tables(self, s):
        out = []
        blocks = re.findall(r"<table[\s\S]*?</table>", s, flags=re.IGNORECASE)
        for t in blocks:
            try:
                for df in pd.read_html(StringIO(t)):
                    arr = [[self._norm_cell(str(x)) for x in row] for row in df.fillna("").astype(str).values.tolist()]
                    if arr:
                        out.append(arr)
            except Exception:
                pass

        # fallback for broken html fragments
        rows = []
        for rb in re.findall(r"<tr[^>]*>(.*?)</tr>", s, flags=re.IGNORECASE | re.DOTALL):
            cells = re.findall(r"<t[dh][^>]*>(.*?)</t[dh]>", rb, flags=re.IGNORECASE | re.DOTALL)
            cells = [self._norm_cell(c) for c in cells if self._norm_cell(c)]
            if len(cells) >= 2:
                rows.append(cells)
        if rows:
            out.append(rows)
        return out

    def _extract_tables(self, s):
        return self._extract_markdown_tables(s) + self._extract_html_tables(s)

    def _flatten_cells(self, tables):
        return [c for t in tables for row in t for c in row if c]

    def _table_score(self, pred_cells, gt_cells):
        if len(gt_cells) == 0:
            return None
        if len(pred_cells) == 0:
            return 0.0
        pc, gc = Counter(pred_cells), Counter(gt_cells)
        overlap = sum((pc & gc).values())
        p = overlap / sum(pc.values()) if pc else 0.0
        r = overlap / sum(gc.values()) if gc else 0.0
        return 0.0 if (p + r) == 0 else 2 * p * r / (p + r)

    # Text
    def _extract_text_only(self, s):
        s = re.sub(r"\$\$(.+?)\$\$", " ", s, flags=re.DOTALL)
        s = re.sub(r"\$(.+?)\$", " ", s, flags=re.DOTALL)
        s = re.sub(r"<table[\s\S]*?</table>", " ", s, flags=re.IGNORECASE)
        s = re.sub(r"<tr[\s\S]*?</tr>", " ", s, flags=re.IGNORECASE)
        s = re.sub(r"(?m)^\s*\|.*\|\s*$", " ", s)
        s = re.sub(r"(?m)^\s*\|?\s*[:\-]+(?:\s*\|\s*[:\-]+)+\s*\|?\s*$", " ", s)
        s = re.sub(r"(?m)^\s*#{1,6}\s*", "", s)
        s = re.sub(r"<[^>]+>", " ", s)
        return re.sub(r"\s+", " ", s).strip()

    # Metrics
    def _edit_distance(self, a, b):
        n, m = len(a), len(b)
        if n == 0:
            return m
        if m == 0:
            return n
        dp = list(range(m + 1))
        for i in range(1, n + 1):
            prev = dp[0]
            dp[0] = i
            for j in range(1, m + 1):
                temp = dp[j]
                cost = 0 if a[i - 1] == b[j - 1] else 1
                dp[j] = min(dp[j] + 1, dp[j - 1] + 1, prev + cost)
                prev = temp
        return dp[m]

    def _cer(self, pred, gt):
        if jiwer is not None:
            try:
                return float(jiwer.cer(gt, pred))
            except Exception:
                pass
        return 0.0 if len(gt) == 0 and len(pred) == 0 else (1.0 if len(gt) == 0 else self._edit_distance(pred, gt) / len(gt))

    def _wer(self, pred, gt):
        if jiwer is not None:
            try:
                return float(jiwer.wer(gt, pred))
            except Exception:
                pass
        pw, gw = pred.split(), gt.split()
        return 0.0 if len(gw) == 0 and len(pw) == 0 else (1.0 if len(gw) == 0 else self._edit_distance(pw, gw) / len(gw))

    def _formula_cer(self, pred_f, gt_f):
        n = min(len(pred_f), len(gt_f))
        if n == 0:
            return 0.0 if len(pred_f) == len(gt_f) else 1.0
        vals = [self._cer(pred_f[i], gt_f[i]) for i in range(n)]
        unmatched = abs(len(pred_f) - len(gt_f))
        return (sum(vals) + unmatched) / (n + unmatched)

    def _bleu(self, pred, gt, max_n=4, smooth=1.0):
        hyp, ref = pred.split(), gt.split()
        if len(hyp) == 0:
            return 0.0
        if len(ref) == 0:
            return 1.0 if len(hyp) == 0 else 0.0

        ps = []
        for n in range(1, max_n + 1):
            h = Counter(tuple(hyp[i:i + n]) for i in range(max(0, len(hyp) - n + 1)))
            r = Counter(tuple(ref[i:i + n]) for i in range(max(0, len(ref) - n + 1)))
            overlap = sum((h & r).values())
            total = sum(h.values())
            ps.append((overlap + smooth) / (total + smooth) if total > 0 else 0.0)

        geo = math.exp(sum(math.log(p) for p in ps) / max_n)
        bp = 1.0 if len(hyp) > len(ref) else math.exp(1 - len(ref) / max(len(hyp), 1))
        return bp * geo

    def _meteor(self, pred, gt):
        hyp, ref = pred.split(), gt.split()
        if len(hyp) == 0 and len(ref) == 0:
            return 1.0
        if len(hyp) == 0 or len(ref) == 0:
            return 0.0
        if meteor_score is not None:
            try:
                return float(meteor_score([ref], hyp))
            except Exception:
                pass
        overlap = sum((Counter(hyp) & Counter(ref)).values())
        p, r = overlap / len(hyp), overlap / len(ref)
        return 0.0 if (p == 0 and r == 0) else (10 * p * r) / (r + 9 * p)

    def _token_f1(self, pred, gt):
        hyp, ref = pred.split(), gt.split()
        if len(hyp) == 0 and len(ref) == 0:
            return 1.0
        if len(hyp) == 0 or len(ref) == 0:
            return 0.0
        overlap = sum((Counter(hyp) & Counter(ref)).values())
        p, r = overlap / len(hyp), overlap / len(ref)
        return 0.0 if (p + r) == 0 else 2 * p * r / (p + r)

    def _ned(self, pred, gt):
        m = max(len(pred), len(gt))
        if m == 0:
            return 1.0
        return max(0.0, 1.0 - self._edit_distance(pred, gt) / m)

    def _final_score(self, long_text_hybrid, table_score, formula_cer):
        comp = {
            "long_text": long_text_hybrid,
            "table": table_score,
            "formula": max(0.0, 1.0 - formula_cer),
        }
        active = {k: v for k, v in comp.items() if v is not None}
        wsum = sum(self.weights[k] for k in active)
        if wsum == 0:
            return 0.0
        return sum((self.weights[k] / wsum) * float(v) for k, v in active.items())

PRED_FILE = Path("/media/data3/users/huytq/huy/outputs/Public039/Public039.md")
GT_FILE = Path("/media/data3/users/huytq/huy/ground_truth/Public039.md")

if not PRED_FILE.exists():
    raise FileNotFoundError(f"Khong tim thay prediction file: {PRED_FILE}")
if not GT_FILE.exists():
    raise FileNotFoundError(f"Khong tim thay ground-truth file: {GT_FILE}")

evaluator = OCRBenchEvaluator(lowercase=True)
result, summary_df = evaluator.evaluate_files(PRED_FILE, GT_FILE)

print("=== OCRBench Evaluation Summary ===")
print(f"CER            : {result['cer']:.4f}")
print(f"WER            : {result['wer']:.4f}")
print(f"BLEU           : {result['bleu']:.4f}")
print(f"METEOR         : {result['meteor']:.4f}")
print(f"F1             : {result['f1']:.4f}")
print(f"NED            : {result['ned']:.4f}")
print(f"LongTextHybrid : {result['long_text_hybrid']:.4f}")
if result['teds_table'] is None:
    print("TEDS_Table     : N/A (GT khong parse duoc bang)")
else:
    print(f"TEDS_Table     : {result['teds_table']:.4f}")
print(f"Formula-CER    : {result['formula_cer']:.4f}")
print(f"Final_Score    : {result['final_score']:.4f}")
print(f"Pred tables/cells: {result['pred_table_count']}/{result['pred_table_cells']}")
print(f"GT   tables/cells: {result['gt_table_count']}/{result['gt_table_cells']}")

print("\n=== DataFrame ===")
display(summary_df)

=== OCRBench Evaluation Summary ===
CER            : 0.1052
WER            : 0.0707
BLEU           : 0.8966
METEOR         : 0.9715
F1             : 0.9544
NED            : 0.9031
LongTextHybrid : 0.9314
TEDS_Table     : N/A (GT khong parse duoc bang)
Formula-CER    : 0.0000
Final_Score    : 0.9543
Pred tables/cells: 3/208
GT   tables/cells: 0/0

=== DataFrame ===


,CER,WER,TEDS_Table,Formula-CER,BLEU,METEOR,F1,NED,LongTextHybrid,Final_Score,Pred-Tables,GT-Tables,Pred-Cells,GT-Cells
0,0.105249,0.070707,<NA>,0.0,0.896568,0.971535,0.954407,0.903119,0.931407,0.954272,3,0,208,0


In [3]:
import torch
import pypdfium2 as pdfium # thư viện để đọc và render PDF
from surya.foundation import FoundationPredictor
from surya.layout import LayoutPredictor
from surya.settings import settings as surya_settings

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print('Device:', device)

layout_model = LayoutPredictor(
    FoundationPredictor(
        checkpoint=surya_settings.LAYOUT_MODEL_CHECKPOINT,
        device=device,
    )
)

pdf_doc = pdfium.PdfDocument(str(pdf_path))
page_count = len(pdf_doc)
run_pages = min(max_pages, page_count)

images = []
for page_id in range(run_pages):
    page = pdf_doc[page_id]
    # scale=96/72 ~ 96 DPI, giong lowres image trong Marker
    pil_image = page.render(scale=96 / 72).to_pil()
    images.append(pil_image)

layout_results = layout_model(images, batch_size=min(6, len(images)))

print(f'Processed {len(layout_results)} / {page_count} pages')
for i, result in enumerate(layout_results):
    labels = [b.label for b in result.bboxes]
    uniq = sorted(set(labels))
    print(f'Page {i}: {len(labels)} blocks | labels={uniq}')

Device: cpu


Recognizing Layout: 100%|██████████| 3/3 [00:31<00:00, 10.44s/it]

Processed 3 / 12 pages
Page 0: 11 blocks | labels=['SectionHeader', 'Text']
Page 1: 14 blocks | labels=['Picture', 'SectionHeader', 'Text']
Page 2: 14 blocks | labels=['SectionHeader', 'Text']


In [4]:
from pathlib import Path
import fitz
import pdfplumber

INPUT_PDF = Path("/media/data3/users/huytq/huy/input/Public278.pdf")
OUTPUT_PDF = INPUT_PDF.with_name(f"{INPUT_PDF.stem}_cropped.pdf")
BUFFER_RATIO = 0.005  

if not INPUT_PDF.exists():
    raise FileNotFoundError(f"Khong tim thay file: {INPUT_PDF}")


def find_crop_ratio_with_pdfplumber(pdf_path: Path, buffer_ratio: float = 0.005):
    with pdfplumber.open(str(pdf_path)) as pdf:
        if not pdf.pages:
            return None
        first_page = pdf.pages[0]
        tables = first_page.find_tables()
        if not tables:
            return None

        # bbox = (x0, top, x1, bottom)
        first_table_bbox = tables[0].bbox
        page_height = first_page.height
        table_bottom = first_table_bbox[3]
        crop_ratio = (table_bottom / page_height) + buffer_ratio
        return min(max(crop_ratio, 0.0), 0.95)


def crop_pdf_top(input_pdf: Path, output_pdf: Path, crop_ratio: float):
    doc = fitz.open(str(input_pdf))
    for page in doc:
        rect = page.rect
        crop_rect = fitz.Rect(
            rect.x0,
            rect.y0 + rect.height * crop_ratio,
            rect.x1,
            rect.y1,
        )
        page.set_cropbox(crop_rect)
    doc.save(str(output_pdf))
    doc.close()


crop_ratio = find_crop_ratio_with_pdfplumber(INPUT_PDF, BUFFER_RATIO)
if crop_ratio is None:
    crop_ratio = 0.10
    print("Khong tim thay table o trang 1 -> fallback crop_ratio=0.10")

crop_pdf_top(INPUT_PDF, OUTPUT_PDF, crop_ratio)

print("=== CROP HEADER DONE ===")
print(f"Input : {INPUT_PDF}")
print(f"Output: {OUTPUT_PDF}")
print(f"Crop ratio: {crop_ratio:.4f}")

=== CROP HEADER DONE ===
Input : /media/data3/users/huytq/huy/input/Public278.pdf
Output: /media/data3/users/huytq/huy/input/Public278_cropped.pdf
Crop ratio: 0.1243


In [5]:
from pathlib import Path
import subprocess
import sys
import time

PDF_PATH = Path("/media/data3/users/huytq/huy/input/Public278_cropped.pdf")
OUT_ROOT = Path("/media/data3/users/huytq/huy/outputs_marker_single")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

if not PDF_PATH.exists():
    raise FileNotFoundError(f"Khong tim thay file PDF: {PDF_PATH}")

marker_single = Path(sys.executable).parent / "marker_single"
if not marker_single.exists():
    print("Khong tim thay marker_single, dang cai marker-pdf...")
    install_cmd = [sys.executable, "-m", "pip", "install", "marker-pdf"]
    install_result = subprocess.run(install_cmd, capture_output=True, text=True)
    if install_result.returncode != 0:
        print(install_result.stdout[-2000:])
        print(install_result.stderr[-2000:])
        raise RuntimeError("Cai marker-pdf that bai")
    marker_single = Path(sys.executable).parent / "marker_single"

cmd = [
    str(marker_single),
    str(PDF_PATH),
    "--output_dir", str(OUT_ROOT),
    "--output_format", "markdown",
    "--disable_multiprocessing",
]

print("Dang chay Marker cho 1 file PDF...")
print("Lenh:", " ".join(cmd))

start = time.time()
result = subprocess.run(cmd, capture_output=True, text=True)
elapsed = time.time() - start

print(f"Return code: {result.returncode}")
print(f"Thoi gian: {elapsed:.1f}s")

base_name = PDF_PATH.stem
md_path = OUT_ROOT / base_name / f"{base_name}.md"

if result.returncode == 0 and md_path.exists():
    print(f"OK -> Markdown: {md_path}")
    md_text = md_path.read_text(encoding="utf-8", errors="ignore")
    print(f"Do dai markdown: {len(md_text)} ky tu")
    print("\nPreview 1000 ky tu dau:")
    print(md_text[:1000])
else:
    print("FAIL: khong tao duoc markdown")
    if result.stderr:
        print(result.stderr[-2000:])
    elif result.stdout:
        print(result.stdout[-2000:])

Dang chay Marker cho 1 file PDF...
Lenh: /media/data3/users/huytq/miniconda3/envs/huy/bin/marker_single /media/data3/users/huytq/huy/input/Public278_cropped.pdf --output_dir /media/data3/users/huytq/huy/outputs_marker_single --output_format markdown --disable_multiprocessing
Return code: 0
Thoi gian: 1199.6s
OK -> Markdown: /media/data3/users/huytq/huy/outputs_marker_single/Public278_cropped/Public278_cropped.md
Do dai markdown: 44991 ky tu

Preview 1000 ky tu dau:
# **1. Tổng quan**

#### **1.1 Mục đích**

Tài liệu trình bày tổng quan giải pháp và quy trình nghiệp vụ đáp ứng cho bài toán tích hợp đối tác kinh doanh tại Tổng công ty cổ phần Bưu chính Viettel.

Thiết kế, mô tả các quy trình nghiệp vụ của Hệ thống đảm bảo cung cấp giải pháp hoàn chỉnh, xuyên suốt quá trình khai báo và phê duyệt mã KH chi COD

#### **1.2 Phạm vi**

| STT | Nghiệp vụ                                 | Phạm vi áp dụng                                                                                            

In [6]:
from pathlib import Path
import re

MD_FILE = Path("/media/data3/users/huytq/huy/outputs_marker_single/Public278_cropped/Public278_cropped.md")

text = MD_FILE.read_text(encoding="utf-8", errors="ignore")
lines = text.split("\n")

# Tach lines thanh blocks
blocks = []
i = 0
while i < len(lines):
    line = lines[i]
    is_table_line = line.strip().startswith("|") and line.strip().endswith("|")
    
    if is_table_line:
        table_lines = [line]
        j = i + 1
        while j < len(lines):
            if lines[j].strip().startswith("|") and lines[j].strip().endswith("|"):
                table_lines.append(lines[j])
                j += 1
            else:
                break
        blocks.append(("table", table_lines))
        i = j
    else:
        text_lines = [line]
        j = i + 1
        while j < len(lines):
            next_line = lines[j]
            if next_line.strip().startswith("|") and next_line.strip().endswith("|"):
                break
            text_lines.append(next_line)
            j += 1
        blocks.append(("text", text_lines))
        i = j

print(f"Tach thanh {len(blocks)} blocks")

# Kiem tra xem nhung cap table nao can ghep
# Ghep khi: text giua 2 tables chi la whitespace (khong co content)
ghep_pairs = []
for i in range(len(blocks) - 1):
    if blocks[i][0] == "table" and blocks[i + 1][0] == "text":
        text_between = "\n".join(blocks[i + 1][1])
        # Chi ghep neu text giua khong co noi dung (chi whitespace/empty)
        if not text_between.strip():
            if i + 2 < len(blocks) and blocks[i + 2][0] == "table":
                ghep_pairs.append((i, i + 2))

print(f"Tong cap can ghep: {len(ghep_pairs)}")
if ghep_pairs:
    for idx1, idx2 in ghep_pairs:
        print(f"  Ghep table {idx1} va table {idx2}")

# Xac dinh separator line (chi chua | va dau -)
def is_separator_line(line):
    return bool(re.match(r"^\s*\|[\s\-|:]+\|\s*$", line.strip()))

# Thuc hien ghep
final_blocks = list(blocks)
for table1_idx, table2_idx in reversed(ghep_pairs):
    table1_lines = final_blocks[table1_idx][1]
    table2_lines = final_blocks[table2_idx][1]
    
    # Xoa dong separator cuoi cua table1 neu co
    while table1_lines and is_separator_line(table1_lines[-1]):
        table1_lines.pop()
    
    # Tim separator line trong table2
    separator_idx = -1
    for idx, line in enumerate(table2_lines):
        if is_separator_line(line):
            separator_idx = idx
            break
    
    if separator_idx >= 0:
        data_rows = table2_lines[separator_idx + 1:]
        merged_lines = table1_lines + data_rows
    else:
        merged_lines = table1_lines + table2_lines
    
    final_blocks[table1_idx] = ("table", merged_lines)
    final_blocks[table1_idx + 1] = ("delete", None)
    final_blocks[table2_idx] = ("delete", None)

# Loai bo cac blocks "delete"
final_blocks = [b for b in final_blocks if b[0] != "delete"]

# Reconstruct output
output_lines = []
for block_type, block_data in final_blocks:
    if block_type == "text":
        output_lines.extend(block_data)
    elif block_type == "table":
        output_lines.extend(block_data)

output_text = "\n".join(output_lines)
output_path = MD_FILE.with_name(f"{MD_FILE.stem}_table_merged.md")
output_path.write_text(output_text + "\n", encoding="utf-8")

print(f"\n=== KET QUA ===")
print(f"Output: {output_path}")
print(f"Do dai goc: {len(text)} ky tu")
print(f"Do dai sau ghep: {len(output_text)} ky tu")
print(f"Tiet kiem: {len(text) - len(output_text)} ky tu")

Tach thanh 23 blocks
Tong cap can ghep: 6
  Ghep table 3 va table 5
  Ghep table 7 va table 9
  Ghep table 9 va table 11
  Ghep table 13 va table 15
  Ghep table 15 va table 17
  Ghep table 19 va table 21

=== KET QUA ===
Output: /media/data3/users/huytq/huy/outputs_marker_single/Public278_cropped/Public278_cropped_table_merged.md
Do dai goc: 44991 ky tu
Do dai sau ghep: 39663 ky tu
Tiet kiem: 5328 ky tu


In [16]:
from pathlib import Path

MD_FILE = Path("/media/data3/users/huytq/huy/viettel_pipeline_test_public283/output/Public283_cropped/Public283_cropped.md")

text = MD_FILE.read_text(encoding="utf-8", errors="ignore")
lines = text.split("\n")

# Tim dau tien cua "Bảng", "table", "<table", "|" de xem co bang khong
print("=== KIEM TRA DINH DANG BANG ===")
print(f"Tong so dong: {len(lines)}\n")

# Tim nhung dong chua "table" (case-insensitive)
table_lines = [i for i, l in enumerate(lines) if "table" in l.lower()]
print(f"Tim thay {len(table_lines)} dong chua 'table'")
if table_lines:
    for idx in table_lines[:5]:
        print(f"  Dong {idx}: {lines[idx][:100]}")

# Tim nhung dong chua "|" (markdown table)
pipe_lines = [i for i, l in enumerate(lines) if "|" in l and l.strip()]
print(f"\nTim thay {len(pipe_lines)} dong chua '|' (markdown table)")
if pipe_lines:
    for idx in pipe_lines[:5]:
        print(f"  Dong {idx}: {lines[idx][:100]}")

# Tim nhung dong chua "<table"
html_table_lines = [i for i, l in enumerate(lines) if "<table" in l.lower()]
print(f"\nTim thay {len(html_table_lines)} dong chua '<table'")
if html_table_lines:
    for idx in html_table_lines[:5]:
        print(f"  Dong {idx}: {lines[idx][:100]}")

# Tim nhung dong chua "|<image" (placeholder images)
image_lines = [i for i, l in enumerate(lines) if "|<image" in l]
print(f"\nTim thay {len(image_lines)} dong chua image placeholder")
if image_lines:
    for idx in image_lines[:5]:
        print(f"  Dong {idx}: {lines[idx][:100]}")

print(f"\n=== PREVIEW xung quanh cac dau hieu ===")

# Lay 5-10 dong dau tien sau heading
heading_lines = [i for i, l in enumerate(lines) if l.strip().startswith("#")]
if heading_lines:
    start_preview = heading_lines[0]
    end_preview = min(start_preview + 15, len(lines))
    print(f"Dau tien sau heading (dong {start_preview}-{end_preview}):")
    for idx in range(start_preview, end_preview):
        print(f"  {idx}: {lines[idx][:80]}")

=== KIEM TRA DINH DANG BANG ===
Tong so dong: 253

Tim thay 4 dong chua 'table'
  Dong 152: |     |                            | ISO 32000-<br>1:2008 | Document<br>management<br>-<br>Portable<
  Dong 154: |     |                            | ISO 32000-<br>2:2020 | Document<br>management<br>-<br>Portable<
  Dong 199: | 3.6 | Định<br>dạng PDF                                                  | ISO 32000-<br>1:2008    
  Dong 200: |     |                                                                   | ISO 32000-<br>2:2020    

Tim thay 84 dong chua '|' (markdown table)
  Dong 137: | Số<br>TT | Loại tiêu chuẩn                                                                        
  Dong 138: |----------|----------------------------------------------------------------------------------------
  Dong 139: | tra chữ  | 1. Tiêu chuẩn bắt buộc áp dụng cho chữ<br>ký số, chứng thư chữ<br>ký số<br>trên thông đ
  Dong 140: | 1.1      | Mật mã đối xứng                                             

In [9]:
def apply_row_window(header, rows, doc_id, table_id, row_window, row_overlap, max_chars, join_str, suffix):
    chunks = []
    i = 0 
    while i < len(rows):
        current_batch = []
        char_count = len(header) + len(suffix)
        end_row = i

        for j in range(i, min(i + row_window, len(rows))):
            row_len = len(rows[j]) + len(join_str)
            if char_count + row_len > max_chars and len(current_batch) > 0:
                break
            current_batch.append(rows[j])
            char_count += row_len 
            end_row = j
        
        chunk_content = header + join_str.join(current_batch) + suffix
        chunks.append({
            "page_content": chunk_content,
            "meta_data": {
                "doc_id": doc_id, 
                "chunk_type": "table",
                "table_id": table_id,
                "row_start": i, 
                "row_end": end_row
            }
        })

        if end_row >= len(rows) - 1:
            break

        step = (end_row - i + 1) - row_overlap
        i += max(1, step)

    return chunks

In [13]:
from bs4 import BeautifulSoup

html = """<table><tbody><tr><th>Số<br/>TT</th><th>Loại tiêu chuẩn</th><th>Ký hiệu tiêu chuẩn</th><th>Tên đầy đủ<br/>của tiêu chuẩn</th><th>Quy định áp dụng</th></tr><tr><td>tra chữ</td><td colspan="6">1. Tiêu chuẩn bắt buộc áp dụng cho chữ<br/>ký số, chứng thư chữ<br/>ký số<br/>trên thông điệp dữ<br/>liệu dùng cho phần mềm ký số<br/>và phần mềm kiểm<br/>ký số</td></tr><tr><td>1.1</td><td>Mật mã đối xứng</td><td>TCVN 7816:2007</td><td>Công nghệ<br/>thông tin -<br/>Kỹ<br/>thuật mật mã<br/>thuật toản mã dũ liệu AES</td><td>Áp dụng một trong hai tiêu chuẩn</td></tr><tr><td></td><td></td><td>FIPS PUB 197</td><td>Advanced Encryption Standard</td><td></td></tr><tr><td>1.2</td><td>Mật mã phi đối<br/>xứng và chữ<br/>ký</td><td>PKCS<br/>#1<br/>(RFC<br/>3447)</td><td>RSA Cryptography Standard</td><td>-<br/>Áp dụng một trong hai tiêu chuẩn<br/>-Đối với tiêu chuẩn RSA:</td></tr><tr><td></td><td>số</td><td>ANSI X9.62-2005</td><td>Public<br/>Key<br/>Cryptography<br/>for<br/>the<br/>Financial Services Industry: The Elliptic<br/>Curve<br/>Digital<br/>Signature<br/>Algorithm<br/>(ECDSA)</td><td>+ Tối thiểu phiên bản 2.1:<br/>+ Áp dụng lược đồ<br/>RSAES-OAEP để<br/>mã<br/>hoá và RSASSA-PSS để<br/>ký.<br/>+ Độ<br/>dài khóa tối thiểu là 2048 bit<br/>-<br/>Đối với tiêu chuẩn ECDSA: độ<br/>dài khóa<br/>tối thiểu là 256 bit.</td></tr><tr><td>1.3</td><td>Yêu cầu cho hàm<br/>băm</td><td>FIPS PUB 180-4<br/>FIPS PUB 202</td><td>Secure Hash Standard<br/>SHA-3<br/>Standard:<br/>Permutation-Based<br/>Hash and Extendable-Output Functions</td><td>Áp dụng một trong các hàm băm sau:<br/>SHA-256,<br/>SHA-384,<br/>SHA-512,<br/>SHA<br/>512/224,<br/>SHA-512/256,<br/>SHA3-224,<br/>SHA3-256,<br/>SHA3-384,<br/>SHA3-512,<br/>SHAKE128, SHAKE256</td></tr><tr><td>1.4</td><td>Cú<br/>pháp<br/>thông<br/>điệp mật mã</td><td>PKCS #7<br/>(RFC 2630)</td><td>Cryptographic Message Syntax Standard</td><td>Phiên bản 1.5</td></tr><tr><td>1.5</td><td>Chứ<br/>ký số<br/>cho</td><td>ETSI EN 319 142-1</td><td>Electronic Signatures and Infrastructures</td><td>Áp dụng một trong hai bộ<br/>tiêu chuẩn: ETSI</td></tr><tr><td></td><td>PDF</td><td></td><td>(ESI);</td><td>EN 319 142-1 Phiên bản V1.2.1</td></tr><tr><td></td><td></td><td></td><td>PAdES<br/>digital<br/>signatures;<br/>Part<br/>1:<br/>Building blocks and PAdES baseline<br/>signatures</td><td>ETSI EN 319 142-2 Phiên bản V1.1.1<br/>Hoặc<br/>ISO 14533-3:2017</td></tr><tr><td></td><td></td><td>ETSI EN 319 142-2</td><td>Electronic Signatures and Infrastructures<br/>(ESI); PAdES digital signatures; Part 2:<br/>Additional PAdES signatures profiles</td><td>ISO 32000-1:2008<br/>ISO 32000-2:2020</td></tr><tr><td></td><td></td><td>ISO 32000-<br/>1:2008</td><td>Document<br/>management<br/>-<br/>Portable<br/>document format -<br/>Part 1: PDF 1.7</td><td></td></tr><tr><td></td><td></td><td>ISO 14533-<br/>3:2017</td><td>Processes, data elements and documents<br/>in<br/>commerce,<br/>industry<br/>and<br/>administration —<br/>Longterm signature<br/>formats —<br/>Part 3: Long-term signature<br/>profiles for PDF Advanced Electronic<br/>Signatures (PAdES)</td><td></td></tr><tr><td></td><td></td><td>ISO 32000-<br/>2:2020</td><td>Document<br/>management<br/>-<br/>Portable<br/>document format -<br/>Part 2: PDF 2.0</td><td></td></tr><tr><td>1.6</td><td>Chữ<br/>ký số<br/>cho<br/>XML</td><td>ETSI EN 319 132-1</td><td>Electronic Signatures and Infrastructures<br/>(ESI); XAdES digital<br/>signatures; Part 1:<br/>Building blocks and XAdES baseline<br/>signatures</td><td>Phiên bản V1.2.1</td></tr><tr><td></td><td></td><td>ETSI EN 319 132-2</td><td>Electronic Signatures and Infrastructures<br/>(ESI); XAdES digital signatures; Part 2:<br/>Extended XAdES signatures</td><td>Phiên bản V1.1.1</td></tr><tr><td>1.7</td><td>Chữ<br/>ký số<br/>cho<br/>CMS</td><td>ETSI EN 319 122-1</td><td>lectronic Signatures and Infrastructures<br/>(ESI); CAdES digital signatures; Part 1:</td><td>Phiên bån V1.3.1</td></tr><tr><td></td><td></td><td></td><td>Building blocks and CAdES baseline<br/>signatures</td><td></td></tr><tr><td></td><td></td><td>ETSI EN 319 122-2</td><td>Electronic Signatures and Infrastructures<br/>(ESI); CAdES digital signatures; Part 2:<br/>Extended CAdES signatures</td><td>Phiên bản V1.1.1</td></tr><tr><td></td><td></td><td>ETSI TS 119 122-3</td><td>Electronic Signatures and Infrastructures<br/>(ESI); CAdES digital signatures; Part 3:<br/>Incorporation<br/>of<br/>Evidence<br/>Record<br/>Syntax (ERS) mechanisms in CAdES</td><td>Phiên bản V1.1.1</td></tr><tr><td>1.8</td><td>Yêu cầu đối với<br/>phần cứng HSM</td><td>FIPS PUB 140-2</td><td>Security Requirements for Cryptographic<br/>Modules</td><td>-<br/>Áp dụng một trong ba tiêu chuẩn.<br/>-<br/>Đối với các tiêu chuẩn FIPS PUB 140-2/</td></tr><tr><td></td><td></td><td>FIPS PUB 140-3</td><td>Security Requirements for Cryptographic<br/>Modules</td><td>FIPS PUB<br/>140-3: Yêu cầu tối<br/>thiểu mức 3 (level 3)</td></tr><tr><td></td><td></td><td>EN 419221-<br/>5:2018</td><td>Protection<br/>Profiles<br/>for<br/>TSP<br/>Cryptographic<br/>modules<br/>-<br/>Part<br/>5:<br/>Cryptographic Module for Trust Services</td><td></td></tr><tr><td>1.9</td><td>Yêu cầu đối với<br/>thẻ<br/>Token<br/>và</td><td>FIPS PUB 140-2</td><td>Security Requirements for Cryptographic<br/>Modules</td><td>-<br/>Áp dụng một trong hai tiêu chuân.<br/>-<br/>Yêu cầu tối thiều mức 2 (level 2)</td></tr><tr><td></td><td>Smart card</td><td>FIPS PUB 140-3</td><td>Security Requirements for Cryptographic<br/>Modules</td><td></td></tr><tr><td>1.10</td><td>Yêu cầu đối với<br/>thẻ<br/>SIM</td><td>FIPS PUB 140-2</td><td>Security Requirements for Cryptographic<br/>Modules</td><td>-<br/>Áp dụng một trong ba tiêu chuẩn.<br/>-<br/>Đối với các tiêu<br/>chuẩn FIPS PUB 140-2/</td></tr><tr><td></td><td></td><td>FIPS PUB 140-3</td><td>Security Requirements for Cryptographic<br/>Modules</td><td>FIPS PUB 140-3: Yêu cầu tối thiểu mức 2<br/>(level 2);</td></tr><tr><td></td><td></td><td>TCVN<br/>8709<br/>(ISO/IEC 15408)</td><td>Công nghệ<br/>thông tin –<br/>Các kỹ<br/>thuật an<br/>toàn –<br/>Các tiêu chí đánh giá an toàn công</td><td>-<br/>Đối với tiêu chuẩn TCVN 8709 (ISO/IEC<br/>15408): Yêu cầu tối thiểu EAL mức 4</td></tr><tr><td></td><td></td><td></td><td>nghệ<br/>thông tin<br/>(Common<br/>Criteria<br/>for<br/>Information<br/>Technology Security Evaluation)</td><td>(level 4)</td></tr><tr><td>1.11</td><td>Yêu<br/>cầu<br/>chính<br/>sách cho Tổ<br/>chức<br/>cung cấp dịch vụ<br/>tạo chữ<br/>|ký số<br/>của<br/>khách hàng</td><td>ETSI TS 119 431-1</td><td>Electronic Signatures and Infrastructures<br/>(ESI); Policy and security requirements<br/>for trust service providers; Part 1: TSP<br/>service components operating a remote<br/>QSCD/SCDev</td><td>Phiên bản V1.2.1</td></tr><tr><td></td><td></td><td>ETSI TS 119 431-2</td><td>Electronic Signatures and Infrastructures<br/>(ESI); Policy and security requirements<br/>for trust service providers; Part 2: TSP<br/>service<br/>components supporting AdES<br/>digital signature creation</td><td>Phiên bản V1.2.1</td></tr><tr><td>1.12</td><td>Giao thức tạo chữ<br/>ký số</td><td>ETSI TS 119 432</td><td>Electronic Signatures and Infrastructures<br/>(ESI);<br/>Protocols<br/>for<br/>remote<br/>digital<br/>signature creation</td><td>Phiên bản V1.2.1</td></tr><tr><td>1.13</td><td>Ứng dụng ký trên<br/>máy chủ<br/>ký số</td><td>EN 419241-<br/>1:2018</td><td>Trustworthy Systems Supporting Server<br/>Signing -<br/>Part 1: General system security<br/>requirements</td><td></td></tr><tr><td>1.14</td><td>Yêu cầu cho mô<br/>đun ký số</td><td>EN 419241-<br/>2:2019</td><td>Trustworthy Systems Supporting Server<br/>Signing -<br/>Part 2: Protection Profile for<br/>QSCD for Server Signing</td><td></td></tr><tr><td>1.15</td><td>Yêu cầu đối với<br/>phần cứng HSM</td><td>EN 419221-<br/>5:2018</td><td>Protection<br/>Profiles<br/>for<br/>TSP<br/>Cryptographic<br/>modules<br/>-<br/>Part<br/>5:<br/>Cryptographic Module for Trust Services</td><td></td></tr><tr><td>1.16</td><td>Giao thức truyền, RFC 2585</td><td></td><td>Internet X.509 Public Key Infrastructure Áp dụng một hoặc cả</td><td>hai giao thức FTP và</td></tr><tr><td></td><td>nhận chứng thư<br/>chữ<br/>ký số<br/>và danh<br/>sách chứng thư<br/>chữ<br/>ký số<br/>bị<br/>thu<br/>hồi</td><td></td><td>-<br/>Operational Protocols: FTP and HTTP</td><td>HTTP</td></tr><tr><td>1.17</td><td>Giao<br/>thức<br/>cho<br/>kiểm<br/>tra<br/>trạng<br/>thái<br/>chứng<br/>thu<br/>chữ<br/>ký số<br/>trực<br/>tuyến</td><td>RFC 6960</td><td>X.509 Internet Public Key Infrastructure<br/>Online Certificate Status Protocol -<br/>OCSP</td><td></td></tr><tr><td>1.18</td><td>Định dạng chứng<br/>thư chữ<br/>ký số<br/>và<br/>danh sách thu hồi<br/>chứng thư chữ<br/>ký<br/>số</td><td>RFC 5280</td><td>Internet X.509 Public Key Infrastructure<br/>Certificate and Certificate Revocation<br/>List (CRL) Profile</td><td></td></tr><tr><td></td><td></td><td></td><td>2. Tiêu chuần bắt buộc áp dụng cho phần mềm ký số, phần mềm kiểm tra chữ<br/>ký số</td><td></td></tr><tr><td>2.1</td><td>Bộ<br/>ký tự<br/>và mã<br/>hóa<br/>cho<br/>tiếng<br/>Việt</td><td>TCVN 6909:2001</td><td>TCVN 6909:2001 " Công nghệ<br/>thông<br/>tin-Bộ<br/>mã ký tự<br/>tiếng Việt 16-bit"</td><td>Bắt buộc áp dụng</td></tr><tr><td>2.2</td><td>Giao thức đường<br/>truyền</td><td>RFC 9110</td><td>HTTP Semantics</td><td>HTTP/1.1</td></tr><tr><td>2.3</td><td>Giao<br/>thức<br/>bảo<br/>mật tầng giao vận</td><td>RFC 5246</td><td>The Transport Layer Security (TLS)<br/>Protocol Version 1.2</td><td>Áp dụng tối thiểu TLS 1.2</td></tr><tr><td colspan="6">3. Tiêu chuần khuyến nghị<br/>áp dụng cho phần mềm ký số, phần mềm kiểm tra chứ<br/>ký số</td></tr><tr><td>3.1</td><td>Bộ<br/>ký tự<br/>và mã<br/>hóa</td><td>ASCII</td><td>American Standard Code for Information<br/>Interchange</td><td></td></tr><tr><td></td><td></td><td></td><td></td><td></td></tr><tr><td>3.2</td><td>Trình diễn bộ<br/>ký<br/>tự</td><td>UTF-8</td><td>8-bit Universal Character Set (UCS)/<br/>Unicode Transformation Format</td><td></td></tr><tr><td>3.3</td><td rowspan="2">Ngôn<br/>ngữ<br/>định<br/>dạng thông điệp<br/>dữ<br/>liệu</td><td>XML<br/>v1.0<br/>(5th<br/>Edition)</td><td>Extensible Markup Language version 1.0<br/>(5th Edition)</td><td>Áp dụng tối thiểu XML v1.0</td></tr><tr><td></td><td>XML<br/>v1.1<br/>(2nd<br/>Edition)</td><td>Extensible Markup Language version 1.1</td><td></td></tr><tr><td>3.4</td><td>Định nghĩa các<br/>lược đồ<br/>trong tài<br/>liệu XML</td><td>XML<br/>Schema<br/>version 1.1</td><td>XML Schema version 1.1</td><td></td></tr><tr><td>3.5</td><td>Trao đổi dũ liệu<br/>đặc<br/>tả<br/>tài<br/>liệu<br/>XML</td><td>XML v2.4.2</td><td>XML Metadata Interchange version 2.4.2</td><td></td></tr><tr><td>3.6</td><td>Định<br/>dạng PDF</td><td>ISO 32000-<br/>1:2008</td><td>Document<br/>management<br/>-<br/>Portable<br/>document format -<br/>Part 1: PDF 1.7</td><td>Áp dụng tối thiểu PDF 1.7</td></tr><tr><td></td><td></td><td>ISO 32000-<br/>2:2020</td><td>Document<br/>management<br/>-<br/>Portable<br/>document format -<br/>Part 2: PDF 2.0</td><td></td></tr><tr><td>3.7</td><td>Định dạng JSON</td><td>RFC 7159</td><td>The JavaScript Object Notation (JSON)<br/>Data Interchange Format</td><td></td></tr><tr><td>3.8</td><td>Cú pháp mã hóa<br/>và<br/>cách<br/>xử<br/>lý<br/>thông điệp dữ<br/>liệu</td><td>XML<br/>Encryption<br/>Syntax<br/>and<br/>Processing</td><td>XML Encryption Syntax and Processing</td><td></td></tr><tr><td></td><td>định dạng XML</td><td>XML<br/>Signature<br/>Syntax<br/>and<br/>Processing</td><td>XML Signature Syntax and Processing</td><td></td></tr><tr><td>3.9</td><td>Quản<br/>lý</td><td>khóa XKMS v2.0</td><td>XML Key Management Specification</td><td></td></tr><tr><td></td><td>công khai thông<br/>điệp dũ liệu định<br/>dạng XML</td><td></td><td>version 2.0</td><td></td></tr><tr><td>3.10</td><td>Chũ ký số<br/>cho<br/>JSON</td><td>RFC 7515</td><td>JSON Web Signature (JWS)</td><td></td></tr></tbody></table>"""

soup = BeautifulSoup(html, "html.parser")

trs = soup.find_all("tr")
rows = []

for tr in trs:
    cells = tr.find_all(["th", "td"])

    row = " | ".join(
        cell.get_text(" ", strip=True)
        for cell in cells
    )

    # print(row + "\n")

    rows.append(row)

header = rows[0] + "\n"

print(header)

data = rows[1:]

chunks = apply_row_window(
    header=header,
    rows=data,
    doc_id="doc_test",
    table_id="table_1",
    row_window=3,
    row_overlap=1,
    max_chars=500,
    join_str="\n",
    suffix=""
)

for idx, chunk in enumerate(chunks, 1):
    print(f"\n==== Chunk {idx} ====")
    print(chunk["page_content"])
    print(chunk["meta_data"])

Số TT | Loại tiêu chuẩn | Ký hiệu tiêu chuẩn | Tên đầy đủ của tiêu chuẩn | Quy định áp dụng


==== Chunk 1 ====
Số TT | Loại tiêu chuẩn | Ký hiệu tiêu chuẩn | Tên đầy đủ của tiêu chuẩn | Quy định áp dụng
tra chữ | 1. Tiêu chuẩn bắt buộc áp dụng cho chữ ký số, chứng thư chữ ký số trên thông điệp dữ liệu dùng cho phần mềm ký số và phần mềm kiểm ký số
1.1 | Mật mã đối xứng | TCVN 7816:2007 | Công nghệ thông tin - Kỹ thuật mật mã thuật toản mã dũ liệu AES | Áp dụng một trong hai tiêu chuẩn
 |  | FIPS PUB 197 | Advanced Encryption Standard | 
{'doc_id': 'doc_test', 'chunk_type': 'table', 'table_id': 'table_1', 'row_start': 0, 'row_end': 2}

==== Chunk 2 ====
Số TT | Loại tiêu chuẩn | Ký hiệu tiêu chuẩn | Tên đầy đủ của tiêu chuẩn | Quy định áp dụng
 |  | FIPS PUB 197 | Advanced Encryption Standard | 
1.2 | Mật mã phi đối xứng và chữ ký | PKCS #1 (RFC 3447) | RSA Cryptography Standard | - Áp dụng một trong hai tiêu chuẩn -Đối với tiêu chuẩn RSA:
{'doc_id': 'doc_test', 'chunk_type': 'table', 